# Indigo Flight Booking Assistant with LangGraph

A conversational AI assistant for booking Indigo airline flights using LangGraph and SQLite database integration.

## Section 1: Import Required Libraries and Set Up LangGraph

In [7]:
pip install -r requirements.txt

  Using cached langgraph-1.0.7-py3-none-any.whl.metadata (7.4 kB)
  Using cached openai-2.16.0-py3-none-any.whl.metadata (29 kB)
  Using cached streamlit-1.53.1-py3-none-any.whl.metadata (10 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.12.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached certifi-2026.1.4-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached 

In [11]:
import sqlite3
import json
import os
from datetime import datetime
from typing import TypedDict, Optional, List, Dict, Any
from enum import Enum
import re

# LangGraph imports
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import create_react_agent

# LangChain imports
#from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


## Section 2: Load and Explore the Indigo Airline Database

In [12]:
# Database setup
DB_PATH = "/Users/vijendra/agentic-ai-usecases/advanced/flight-booking-assistant/indigo_airline.db"

def get_db_connection():
    """Create and return a database connection"""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

# Verify database exists
if os.path.exists(DB_PATH):
    print(f"✅ Database found at: {DB_PATH}")
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print(f"✅ Database contains {len(tables)} tables:")
    for table in tables:
        print(f"  - {table[0]}")
    conn.close()
else:
    print(f"❌ Database not found at: {DB_PATH}")

# Airport code mapping
AIRPORT_CODES = {
    'jaipur': 'JAI',
    'mumbai': 'BOM',
    'delhi': 'DEL',
    'bangalore': 'BLR',
    'hyderabad': 'HYD',
    'kolkata': 'CCU',
    'kochi': 'COK',
    'pune': 'PNQ',
}

AIRPORT_NAMES = {
    'JAI': 'Jaipur International Airport',
    'BOM': 'Chhatrapati Shivaji International Airport',
    'DEL': 'Indira Gandhi International Airport',
    'BLR': 'Kempegowda International Airport',
    'HYD': 'Rajiv Gandhi International Airport',
    'CCU': 'Netaji Subhas Chandra Bose International Airport',
    'COK': 'Cochin International Airport',
    'PNQ': 'Pune Airport',
}

✅ Database found at: /Users/vijendra/agentic-ai-usecases/advanced/flight-booking-assistant/indigo_airline.db
✅ Database contains 17 tables:
  - Customers
  - FlightSchedule
  - DaysOfOperation
  - sqlite_sequence
  - PNRs
  - Bookings
  - Passengers
  - Itineraries
  - ItineraryLegs
  - PassengerBaggage
  - SpecialBaggage
  - FlightInstances
  - FlightDelays
  - Payments
  - Refunds
  - ConnectionRules
  - AuditLog


In [13]:
SYSTEM_PERSONA = """
You are 6ESkai, the official virtual booking assistant of IndiGo Airlines.

Personality:
- Friendly, professional, calm, concise
- Speak exactly like an airline customer support chat
- Never use emojis
- Never use slang

Rules:
- Ask only one question at a time
- Never assume information not provided by the user
- Never invent flight details
- Do not repeat confirmed information
- Dates must be spoken in full format (e.g. 10 February 2026)
"""

GREETING_PROMPT = """
{system}

User message:
"{user_input}"

If the user greets, respond with:
"Hello! I'm 6ESkai, your friendly AI assistant from Indigo.
How can I help you with our services today?
- Book a flight ticket
- Flight Status
- Web Check in"

If the user selects booking, return ONLY this JSON:
{{"intent": "book_flight"}}
"""

EXTRACTION_PROMPT = """
{system}

User message: "{user_input}"

Extract any flight booking information mentioned. If the user mentions a city name, classify it as either departure_city or destination_city based on context. Common Indian cities: Mumbai, Delhi, Bangalore, Chennai, Hyderabad, Pune, Jaipur, Goa, Kolkata, etc.

Return ONLY valid JSON with these optional fields:
{{
    "departure_city": "city name or null",
    "destination_city": "city name or null",
    "travel_date": "date or null",
    "adults": "number or null",
    "children": "number or null"
}}

Return null for any information not mentioned.
"""

CONVERSATION_DRIVER_PROMPT = """
{system}

Current booking state:
{state}

Ask ONLY the next missing question.

If destination_city missing:
"Please let us know your destination"

If departure_city missing:
"Please let us know your starting city"

If travel_date missing:
"Which day is your journey starting on?"

If trip_type missing:
"On which date will you conclude your travels?
- One-way only"

If adults missing:
"Can you please tell me the number of passengers? eg. 2 adults, 1 child"

If children missing:
"Just to confirm how many child passengers are there?
Child age range:
EU region - between 2 and 16 years
Others - between 2 and 12 years
- No child passengers"

If all filled, respond with ONLY:
READY_FOR_CONFIRMATION
"""

CONFIRMATION_PROMPT = """
{system}

Booking details:
{state}

Generate confirmation exactly like IndiGo chat.

If user replies yes → return CONFIRMED
If user replies no → return RESTART
If user edits → return updated fields as JSON
"""

FLIGHT_SELECTION_PROMPT = """
{system}

Available flights:
{flights}

User input:
"{user_input}"

Return ONLY:
{{"selected_flight_id": <number>}}
"""
